# Athena Database

In [1]:
import boto3
import sagemaker

sess = sagemaker.Session()
bucket = sess.default_bucket()
role = sagemaker.get_execution_role()
region = boto3.Session().region_name

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [2]:
s3_parquet_path = "s3://sagemaker-us-east-1-318401170150/curated/ndbc/buoy=46086/"

In [3]:
from pyathena import connect
database_name = "ndbc_data"

In [4]:
# Set S3 staging directory -- this is a temporary directory used for Athena queries
s3_staging_dir = "s3://{0}/athena/staging".format(bucket)

In [5]:
conn = connect(region_name=region, s3_staging_dir=s3_staging_dir)

In [6]:
statement = "CREATE DATABASE IF NOT EXISTS {}".format(database_name)
print(statement)

CREATE DATABASE IF NOT EXISTS ndbc_data


In [7]:
import pandas as pd

pd.read_sql(statement, conn)

/tmp/ipykernel_2042/3803073958.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(statement, conn)


""


In [8]:
statement = "SHOW DATABASES"

df_show = pd.read_sql(statement, conn)
df_show.head(5)

/tmp/ipykernel_2042/3999478089.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_show = pd.read_sql(statement, conn)


,database_name
0,default
1,dsoaws
2,ndbc
3,ndbc_data
4,sagemaker_featurestore


In [9]:
table_name = "sensor_values"

In [10]:
# SQL statement to execute
statement = f"""
CREATE EXTERNAL TABLE IF NOT EXISTS {database_name}.{table_name} (
    `timestamp` timestamp,
    station_id string,
    wind_direction double,
    wind_speed double,
    wind_gust double,
    wave_height double,
    dominant_wave_period double,
    average_wave_period double,
    mean_wave_direction double,
    pressure double,
    air_temperature double,
    water_temperature double,
    dewpoint_temperature double,
    tide double,
    wind_speed_ms double,
    wave_energy double
)
STORED AS PARQUET
LOCATION '{s3_parquet_path}'
"""

print(statement)


CREATE EXTERNAL TABLE IF NOT EXISTS ndbc_data.sensor_values (
    `timestamp` timestamp,
    station_id string,
    wind_direction double,
    wind_speed double,
    wind_gust double,
    wave_height double,
    dominant_wave_period double,
    average_wave_period double,
    mean_wave_direction double,
    pressure double,
    air_temperature double,
    water_temperature double,
    dewpoint_temperature double,
    tide double,
    wind_speed_ms double,
    wave_energy double
)
STORED AS PARQUET
LOCATION 's3://sagemaker-us-east-1-318401170150/curated/ndbc/buoy=46086/'



In [11]:
import pandas as pd

pd.read_sql(statement, conn)

/tmp/ipykernel_2042/3803073958.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(statement, conn)


""


In [12]:
statement = "SHOW TABLES in {}".format(database_name)

df_show = pd.read_sql(statement, conn)
df_show.head(5)

/tmp/ipykernel_2042/2201015668.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_show = pd.read_sql(statement, conn)


,tab_name
0,ndbc_stdmet
1,sensor_values


In [13]:
# Sample query

statement = """SELECT * FROM {}.{} LIMIT 5""".format(
    database_name, table_name
)

print(statement)

df = pd.read_sql(statement, conn)
df.head(5)

SELECT * FROM ndbc_data.sensor_values LIMIT 5


/tmp/ipykernel_2042/1129130929.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(statement, conn)


,timestamp,station_id,wind_direction,wind_speed,wind_gust,wave_height,dominant_wave_period,average_wave_period,mean_wave_direction,pressure,air_temperature,water_temperature,dewpoint_temperature,visibility,ptdy,tide,wind_speed_ms,wave_energy,buoy
0,2023-01-01 00:10:00,46086,172.0,4.7,5.9,1.63,11.43,9.33,283.0,1014.2,14.9,15.9,13.8,None,None,None,2.417887,30.368367,None
1,2023-01-01 00:40:00,46086,162.0,5.2,6.6,1.88,13.79,9.55,257.0,1014.0,14.9,15.9,14.0,None,None,None,2.675109,48.739376,None
2,2023-01-01 01:10:00,46086,154.0,5.8,7.2,1.68,12.12,8.98,271.0,1013.5,14.9,15.9,14.1,None,None,None,2.983775,34.207488,None
3,2023-01-01 01:40:00,46086,162.0,6.2,7.3,1.80,12.12,8.99,270.0,1013.1,15.0,15.9,14.3,None,None,None,3.189553,39.268800,None
4,2023-01-01 02:10:00,46086,176.0,7.0,8.2,1.76,11.43,8.94,263.0,1012.7,15.0,15.9,14.3,None,None,None,3.601108,35.405568,None


## Release Resources

In [14]:
%%javascript

try {
    Jupyter.notebook.save_checkpoint();
    Jupyter.notebook.session.delete();
}
catch(err) {
    // NoOp
}

<IPython.core.display.Javascript object>

In [15]:
%%html

<p><b>Shutting down your kernel for this notebook to release resources.</b></p>
<button class="sm-command-button" data-commandlinker-command="kernelmenu:shutdown" style="display:none;">Shutdown Kernel</button>
        
<script>
try {
    els = document.getElementsByClassName("sm-command-button");
    els[0].click();
}
catch(err) {
    // NoOp
}    
</script>